In [1]:
# =====================================
# 0. 라이브러리 불러오기
# =====================================
import pandas as pd
import re   # 텍스트 정제용

# 데이터 나누기
from sklearn.model_selection import train_test_split

# 텍스트 → 숫자로 변환
from sklearn.feature_extraction.text import TfidfVectorizer

# 오늘 사용할 모델 (SGDClassifier)
from sklearn.linear_model import SGDClassifier

# 성능 평가
from sklearn.metrics import accuracy_score


# =====================================
# 1. 데이터 불러오기
# =====================================
df = pd.read_csv("data/mbti_1.csv")

print("데이터 크기:", df.shape)
print(df.head())


# =====================================
# 2. 텍스트 전처리 함수 만들기
# =====================================
# 머신러닝은 깨끗한 데이터를 좋아함
# → 불필요한 요소 제거

def clean_text(text):
    # 1. URL 제거 (유튜브 링크 등)
    text = re.sub(r'http\S+', '', text)

    # 2. 영어 알파벳만 남기기 (숫자, 특수문자 제거)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. 소문자로 변환 (Love → love)
    text = text.lower()

    return text


# 모든 데이터에 적용
df['posts'] = df['posts'].apply(clean_text)


# =====================================
# 3. 입력(X) / 정답(y) 나누기
# =====================================
# X = 우리가 분석할 데이터 (텍스트)
# y = 맞춰야 할 정답 (MBTI)
X = df['posts']
y = df['type']


# =====================================
# 4. Train / Test 분리
# =====================================
# test_size=0.2 → 20%는 테스트용
# random_state=42 → 항상 같은 결과 나오게 고정 (중요)
# stratify=y → MBTI 비율 유지
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =====================================
# 5. Train / Validation 분리
# =====================================
# train 데이터를 다시 나눠서 검증 데이터 생성
# 최종 비율: train 60% / valid 20% / test 20%
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

print("\n데이터 분리 결과")
print("Train:", len(X_train))
print("Valid:", len(X_valid))
print("Test :", len(X_test))


# =====================================
# 6. 텍스트 → 숫자로 변환 (TF-IDF)
# =====================================
# 머신러닝은 숫자만 이해하기 때문에
# 텍스트를 숫자로 바꿔야 함

vectorizer = TfidfVectorizer(
    max_features=5000,     # 중요한 단어 5000개만 사용
    stop_words='english',  # 의미 없는 단어 제거 (the, is 등)
    ngram_range=(1, 2)     # 단어 1개 + 2개 묶음까지 사용
)

# ⭐ 매우 중요
# fit = 단어를 학습
# transform = 숫자로 변환

# 학습 데이터로만 fit 해야 함 (데이터 누수 방지)
X_train_vec = vectorizer.fit_transform(X_train)

# 검증 / 테스트는 transform만
X_valid_vec = vectorizer.transform(X_valid)
X_test_vec = vectorizer.transform(X_test)


# =====================================
# 7. 모델 생성 (SGDClassifier)
# =====================================
# SGD = 데이터를 조금씩 보면서 학습하는 방식
# → 속도가 빠르고 텍스트 데이터에 강함

model = SGDClassifier(
    loss='hinge',       # SVM 방식 (분류 문제에서 많이 사용)
    max_iter=1000,      # 반복 횟수
    random_state=42     # 결과 재현성
)


# =====================================
# 8. 모델 학습
# =====================================
model.fit(X_train_vec, y_train)


# =====================================
# 9. 검증 데이터로 성능 확인
# =====================================
# 모델이 얼마나 잘 맞추는지 확인

y_valid_pred = model.predict(X_valid_vec)

valid_acc = accuracy_score(y_valid, y_valid_pred)

print("\n검증 정확도:", valid_acc)


# =====================================
# 10. 테스트 데이터로 최종 평가
# =====================================
y_test_pred = model.predict(X_test_vec)

test_acc = accuracy_score(y_test, y_test_pred)

print("테스트 정확도:", test_acc)


# =====================================
# 11. 새로운 문장 예측해보기 (재미용🔥)
# =====================================
new_text = ["I enjoy deep conversations and thinking alone"]

# 같은 방식으로 변환
new_vec = vectorizer.transform(new_text)

# 예측
print("\n새 문장 예측:", model.predict(new_vec))

데이터 크기: (8675, 2)
   type                                              posts
0  INFJ  'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1  ENTP  'I'm finding the lack of me in these posts ver...
2  INTP  'Good one  _____   https://www.youtube.com/wat...
3  INTJ  'Dear INTP,   I enjoyed our conversation the o...
4  ENTJ  'You're fired.|||That's another silly misconce...

데이터 분리 결과
Train: 5205
Valid: 1735
Test : 1735

검증 정확도: 0.6201729106628242
테스트 정확도: 0.6097982708933718

새 문장 예측: ['INTP']


In [ ]:
# DataFrame으로 보기
# 벡터 → DataFrame으로 변환
df_tfidf = pd.DataFrame(
    X_train_vec.toarray(),                   # 숫자 데이터
    columns=vectorizer.get_feature_names_out()  # 단어 리스트
)

# 상위 5개 데이터 보기
print(df_tfidf.head())

In [ ]:
# 특정 문장만 보기
# # 첫 번째 문장 벡터 확인
first_vec = X_train_vec[0].toarray()[0]

# 단어 + 값 출력
for word, score in zip(vectorizer.get_feature_names_out(), first_vec):
    if score > 0:   # 값이 있는 단어만 출력
        print(word, ":", score)

In [ ]:
# 상위 중요 단어만 보기
import numpy as np

first_vec = X_train_vec[0].toarray()[0]

# 상위 10개 단어 뽑기
top_indices = np.argsort(first_vec)[-10:]

for idx in top_indices:
    print(vectorizer.get_feature_names_out()[idx], ":", first_vec[idx])
